### Integral Timescale and Velocity Uncertainty 

The main idea behind the integral timescale is to determine the effective degrees of freedom on the velocity data. Velocity data is inherently dominated by low frequency motion which leads to every sample not being fully indepenent of each other. 

Goal is to get confidence interval / uncertainty estimate for the mean velocity of a timeseries 


Analysis Steps: 

- Determine the integral timescale via autocorrelation 

- Estimate the effective degrees of freedom (N* = N delta T / integral timescale )

Functions to use: 

- integral_timescale_trapezoidal (follows definition of integral timescale in Emery and Thomson)
- standard_error_saunders (follows standard error estimate from Saunders (1987))

It's ok to just report the standard error as the error estimate on the mean velocity. Should try to do this for smaller timescales (i.e. monthly)

In [ ]:
### imports
import numpy as np
import matplotlib.pyplot as plt 
import scipy 
import xarray as xr  
from TNB_tools import Integral_Timescale as its
import TNB_tools.tides.tidal_decomposition as tides

In [ ]:
# # load in data - DELL 
# vel = xr.open_dataset("//thepenguin/penguin2/Data/TNB/mooring/LDEO/2018_19 Araon Cruise (ANA09B) - NIWAMooringRetrieval/Data/TNBD/working/tnbd_2018_vel_bot_1093.nc") 

# load data - MAC 
vel = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/tnbd_2018_vel_bot_1093.nc')

In [ ]:
# LOAD IN INTERMEDIATE DATASETS DELL - 30 MIN AVERAGED DATA 
TNBD_ctd_top = xr.open_dataset('data/TNBD_ctd_top_30min.nc')
TNBD_ctd_bot = xr.open_dataset('data/TNBD_ctd_bot_30min.nc')
TNBD_vel_top = xr.open_dataset('data/TNBD_vel_top_30min.nc')
TNBD_vel_bot = xr.open_dataset('data/TNBD_vel_bot_30min.nc')

DITD_ctd_top = xr.open_dataset('data/DITD_ctd_top_30min.nc')
DITD_ctd_bot = xr.open_dataset('data/DITD_ctd_bot_30min.nc')
DITD_vel_top = xr.open_dataset('data/DITD_vel_top_30min.nc')
DITD_vel_bot = xr.open_dataset('data/DITD_vel_bot_30min.nc')      # sensor cut out in june 2018

D_ctd_top = xr.open_dataset('data/D_ctd_top_30min.nc')
D_ctd_mid = xr.open_dataset('data/D_ctd_mid_30min.nc')
# D_ctd_bot = D_ctd_bot.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out in 2017

# D_vel_top = D_vel_top.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out end of 2017
D_vel_mid = xr.open_dataset('data/D_vel_mid_30min.nc')
D_vel_bot = xr.open_dataset('data/D_vel_bot_30min.nc')

In [ ]:
its.check_gaps(vel)

In [ ]:
# cropping dataset in December 2018 to avoid weird timestamps 
vel = vel.sel(time=slice('2018-03','2018-12'))

In [ ]:
its.check_gaps(vel)

In [ ]:
dt_seconds = 900

In [ ]:
# defining how far out to do the lag for
nlags = int(70 * 86400 / dt_seconds)  # 70 days, in samples
rho = its.autocorrelation(vel.v, nlags=nlags)

In [ ]:
dt_seconds = 900.0
lags_samples = np.arange(len(rho))
lags_hours = lags_samples * dt_seconds / 3600
lags_days = lags_samples * dt_seconds / 86400

In [ ]:
plt.plot(lags_days,rho)
plt.xlabel('Lag (days)')
plt.ylabel('Autocorrelation')
plt.axhline(y=0,color='k',ls='--')

In [ ]:
T_l, zero_crossing = its.integral_timescale_trapezoidal(rho,dt_seconds)

In [ ]:
T_l / 86400

In [ ]:
# standard error
e = its.standard_error_saunders(sigma=np.std(vel.v),T_L=T_l,T_record=(len(vel.v)*dt_seconds))

In [ ]:
np.mean(vel.v)

In [ ]:
e

In [ ]:
effective_dof = its.effective_dof(N=len(vel.u), dt=dt_seconds,T_int=T_l)

In [ ]:
effective_dof

In [ ]:
nlags = int(70 * 86400 / dt_seconds)  # search within 70 days
results = {v: its.analyze_component(vel, v, dt_seconds, nlags=nlags) for v in ("u", "v")}

In [ ]:
its.compare_confidence_intervals(vel['v'].values, dt_seconds)

### Integral timescale on low-passed filtered data 

In [ ]:
lowpass_ds, highpass_ds = tides.decompose_tidal_dataset(
    vel, var_names=["u", "v"], window_days=1, sampling_hours=0.25
)

In [ ]:
lowpass_ds

In [ ]:
lowpass_ds.u.plot(figsize=(12,4))

In [ ]:
# check data 
its.check_gaps(lowpass_ds.u)

In [ ]:
nan_mask = lowpass_ds["u"].isnull()
print(nan_mask.sum().item(), "total NaNs")
print(lowpass_ds["u"].time.where(nan_mask, drop=True).values)  # timestamps of NaNs

In [ ]:
lowpass_clean = lowpass_ds["u"].dropna(dim="time")

In [ ]:
lowpass_clean

In [ ]:
dt_seconds = 900

In [ ]:
nlags = int(70 * 86400 / dt_seconds)  # 10 days, in samples
rho = its.autocorrelation(lowpass_clean, nlags=nlags)

In [ ]:
dt_seconds = 900.0
lags_samples = np.arange(len(rho))
lags_hours = lags_samples * dt_seconds / 3600
lags_days = lags_samples * dt_seconds / 86400

In [ ]:
plt.plot(lags_days,rho)
plt.xlabel('Lag (days)')
plt.ylabel('Autocorrelation')
plt.axhline(y=0,color='k',ls='--')

In [ ]:
T_l, zero_crossing = its.integral_timescale_trapezoidal(rho,dt_seconds)

In [ ]:
# standard error
e = its.standard_error_saunders(sigma=np.std(lowpass_clean),T_L=T_l,T_record=(len(lowpass_clean)*dt_seconds))

In [ ]:
e

In [ ]:
T_l / 86400

### progressive vector diagram 

In [ ]:
import TNB_tools.pvd_tools as pvd

In [ ]:
x,y = pvd.compute_pvd(vel.u/100,vel.v/100,dt=900)

In [ ]:
pvd.plot_pvd(x,y)

In [ ]:
import pandas as pd

In [ ]:
# x, y = compute_pvd(u_ms, v_ms, time=time, distance_units="km")
event_time = pd.Timestamp("2018-07-25T00:00")
event_idx = int(np.argmin(np.abs(pd.to_datetime(vel.time) - event_time)))
x_c, y_c = pvd.recenter_pvd(x, y, event_idx)
look_back = slice(max(0, event_idx - 960), event_idx + 1)  # ~10 days @ 900s
pvd.plot_pvd(x_c[look_back], y_c[look_back], time=vel.time[look_back],
            event_index=-1, event_label="Event",
            title="Backward track, recentered on event")
plt.show()

In [ ]:
# 2. Convert velocity to displacement (distance = velocity * time)
hours = len(vel.u)*900 / 3600
dt = 900 # seconds
dx = vel.u /10000 * dt
dy = vel.v / 10000* dt

# 3. Calculate the cumulative sum to get the total path (x, y)
x_path = np.cumsum(dx)
y_path = np.cumsum(dy)

# 4. Plot the Progressive Vector Diagram
plt.figure(figsize=(8, 8))
plt.plot(x_path, y_path, color='blue', linewidth=1.5, label='Water Parcel Path')

# Mark the starting point
plt.plot(x_path[0], y_path[0], 'go', markersize=8, label='Start')

# # Add daily markers (24 hours * 3600 seconds = 86400 seconds)
# day_indices = np.arange(0, hours, 24)
# plt.plot(x_path[day_indices], y_path[day_indices], 'ro', markersize=4, label='Daily Mark')

# Formatting the plot
plt.title('TNBD Progressive Vector Diagram')
plt.xlabel('Eastward Displacement (km)')
plt.ylabel('Northward Displacement (km)')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend()
plt.axis('equal') # Ensures 1 meter East = 1 meter North visually
plt.show()

In [ ]:
fig = plt.figure(figsize=(10,4))
plt.plot(vel.u.time,vel.u,label='u')
plt.plot(vel.v.time,vel.v,label='v')
plt.legend()
plt.axhline(y=0,color='k',ls='--')
plt.ylabel('Velocity (cm/s)')
plt.xlabel('Time')

### Estimates on all TNBD, DITD, and Mooring D 

In [ ]:
# load in all data 


In [ ]:
import pandas as pd 

In [ ]:
# load in DITD and D data - MAC
DITD_vel = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/vel_1100.nc')

TNBD_vel = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/tnbd_2018_vel_bot_1093.nc')

moorings = xr.open_datatree('/Users/mackenzie/Documents/Research/PhD/TNB_tools/preprocess/TNB_Moorings_DataTree.nc')

In [ ]:
# fix DITD time - should save this change permenantly later
DITD_vel['time'] = DITD_vel.time - pd.Timedelta(12,'h')

In [ ]:
# rename variables in DITD velocity dataset 
DITD_vel = DITD_vel.rename({
    'east':'u',
    'north':'v',
    'up':'w',
    'pressure':'pres',
    'direction':'dir'
}).assign_coords({
    'z':1121,
    'lat':-75.28,
    'lon':164.1
})
# # rename variables in DITD velocity dataset 
# DITD_vel_bot = DITD_vel_bot.rename({
#     'east':'u',
#     'north':'v',
#     'up':'w',
#     'pressure':'pres',
#     'direction':'dir'
# }).assign_coords({
#     'z':1218,
#     'lat':-75.28,
#     'lon':164.1
# })

In [ ]:
D_vel = moorings['D/2017/vel/bot'].to_dataset().sel(time=slice('2018-01-01','2018-12-31'))

TNBD sampling interval is 15 min. DITD sampling interval is 10 min. D sampling interval is 30 min

In [ ]:
DITD_vel['u'] = DITD_vel.u * 100
DITD_vel['v'] = DITD_vel.v * 100

In [ ]:

# Mooring registry — update fs once you confirm sampling intervals
# from ds.time.diff('time').median()
moorings = {
    'TNBD': {'ds': TNBD_vel, 'fs': 1/900,  'color': 'steelblue', 'label': 'TNBD'},
    'DITD': {'ds': DITD_vel, 'fs': 1/600,  'color': 'tomato',    'label': 'DITD'},
    'D':    {'ds': D_vel,    'fs': 1/1800,  'color': 'seagreen',  'label': 'D'},
}

In [ ]:
from TNB_tools.Integral_Timescale import (
    check_gaps,
    autocorrelation,
    integral_timescale_trapezoidal,
    one_sided_integral_timescale,
    effective_dof,
    standard_error_saunders,
    confidence_interval_mean,
)

MAX_LAG_DAYS = 70

timescale_results = {}

for name, m in moorings.items():
    ds = m['ds']
    dt = 1 / m['fs']
    u  = ds['u'].values
    v  = ds['v'].values
    N  = len(u)
    T_record = N * dt   # seconds

    nlags = int(MAX_LAG_DAYS * 86400 / dt)

    rho_u = autocorrelation(u, nlags=nlags)
    rho_v = autocorrelation(v, nlags=nlags)

    # Trapezoidal one-sided integral timescale (E&T convention)
    T_ET_u, k_star_u = integral_timescale_trapezoidal(rho_u, dt)
    T_ET_v, k_star_v = integral_timescale_trapezoidal(rho_v, dt)

    # Standard error on the mean (Saunders 1987)
    # Note: standard_error_saunders expects one-sided T_L
    # integral_timescale_trapezoidal already returns one-sided T, so use directly
    sigma_u   = u.std(ddof=1)
    sigma_v   = v.std(ddof=1)
    se_u      = standard_error_saunders(sigma_u, T_ET_u, T_record)
    se_v      = standard_error_saunders(sigma_v, T_ET_v, T_record)

    # N* for reference
    N_star_u  = effective_dof(N, dt, T_ET_u * 2)   # effective_dof expects two-sided T_int
    N_star_v  = effective_dof(N, dt, T_ET_v * 2)

    mean_u    = u.mean()
    mean_v    = v.mean()

    # 95% CI using normal approximation (Saunders style)
    from scipy.stats import norm
    z95 = norm.ppf(0.975)

    timescale_results[name] = {
        'lags_hours': np.arange(nlags + 1) * dt / 3600,
        'rho_u':      rho_u,
        'rho_v':      rho_v,
        'T_ET_u':     T_ET_u,
        'T_ET_v':     T_ET_v,
        'k_star_u':   k_star_u,
        'k_star_v':   k_star_v,
        'N_star_u':   N_star_u,
        'N_star_v':   N_star_v,
        'se_u':       se_u,
        'se_v':       se_v,
        'mean_u':     mean_u,
        'mean_v':     mean_v,
        'ci_u':       (mean_u - z95 * se_u, mean_u + z95 * se_u),
        'ci_v':       (mean_v - z95 * se_v, mean_v + z95 * se_v),
        'dt':         dt,
        'T_record_days': T_record / 86400,
    }

    print(f"\n{'='*45}")
    print(f"Mooring: {name}  (dt={dt:.0f}s, N={N}, "
          f"T_record={T_record/86400:.1f} days)")
    print(f"{'='*45}")
    for comp, T_ET, k_star, N_star, mean, se, ci in [
        ('u', T_ET_u, k_star_u, N_star_u, mean_u, se_u, timescale_results[name]['ci_u']),
        ('v', T_ET_v, k_star_v, N_star_v, mean_v, se_v, timescale_results[name]['ci_v']),
    ]:
        print(f"  {comp}: T_ET={T_ET/3600:.2f} hr  k*={k_star}  "
              f"N*={N_star:.0f}  mean={mean:.3f}  "
              f"SE={se:.4f}  95%CI=({ci[0]:.3f}, {ci[1]:.3f})")

In [ ]:
seasons = {
    'MAM': [3, 4, 5],
    'JJA': [6, 7, 8],
    'SON': [9, 10, 11],
    'DJF': [12],
}
season_notes = {
    'MAM': 'Mar–May',
    'JJA': 'Jun–Aug',
    'SON': 'Sep–Nov',
    'DJF': 'Dec only',
}
season_order  = ['MAM', 'JJA', 'SON', 'DJF']
mooring_order = list(moorings.keys())

timescale_seasonal = {name: {} for name in mooring_order}

for name in mooring_order:
    ds = moorings[name]['ds']
    dt = 1 / moorings[name]['fs']

    print(f"\n{'='*45}")
    print(f"Mooring: {name}")
    print(f"{'='*45}")

    for season, months in seasons.items():
        seasonal  = ds.sel(time=ds.time.dt.month.isin(months))
        u         = seasonal['u'].values
        v         = seasonal['v'].values
        N         = len(u)
        T_record  = N * dt

        # Need enough lags to find zero crossing but can't exceed record
        nlags = min(int(MAX_LAG_DAYS * 86400 / dt), N // 2)

        if N < 2 * nlags:
            print(f"  {season}: only {N} samples, skipping")
            continue

        if np.any(np.isnan(u)) or np.any(np.isnan(v)):
            print(f"  {season}: NaNs present, skipping")
            continue

        try:
            rho_u = autocorrelation(u, nlags=nlags)
            rho_v = autocorrelation(v, nlags=nlags)

            T_ET_u, k_star_u = integral_timescale_trapezoidal(rho_u, dt)
            T_ET_v, k_star_v = integral_timescale_trapezoidal(rho_v, dt)
        except ValueError as e:
            print(f"  {season}: {e}")
            continue

        sigma_u  = u.std(ddof=1)
        sigma_v  = v.std(ddof=1)
        se_u     = standard_error_saunders(sigma_u, T_ET_u, T_record)
        se_v     = standard_error_saunders(sigma_v, T_ET_v, T_record)
        N_star_u = effective_dof(N, dt, T_ET_u * 2)
        N_star_v = effective_dof(N, dt, T_ET_v * 2)
        mean_u   = u.mean()
        mean_v   = v.mean()

        timescale_seasonal[name][season] = {
            'T_ET_u':    T_ET_u,
            'T_ET_v':    T_ET_v,
            'k_star_u':  k_star_u,
            'k_star_v':  k_star_v,
            'N_star_u':  N_star_u,
            'N_star_v':  N_star_v,
            'se_u':      se_u,
            'se_v':      se_v,
            'mean_u':    mean_u,
            'mean_v':    mean_v,
            'ci_u':      (mean_u - z95 * se_u, mean_u + z95 * se_u),
            'ci_v':      (mean_v - z95 * se_v, mean_v + z95 * se_v),
            'N':         N,
            'T_record_days': T_record / 86400,
        }

        print(f"  {season} ({season_notes[season]}, N={N}):")
        for comp, T_ET, k_star, N_star, mean, se in [
            ('u', T_ET_u, k_star_u, N_star_u, mean_u, se_u),
            ('v', T_ET_v, k_star_v, N_star_v, mean_v, se_v),
        ]:
            print(f"    {comp}: T_ET={T_ET/3600:.2f} hr  k*={k_star}  "
                  f"N*={N_star:.0f}  mean={mean:.3f}  SE={se:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (name, res) in zip(axes, timescale_results.items()):
    color = moorings[name]['color']
    lags  = res['lags_hours']

    ax.plot(lags, res['rho_u'], color=color, linewidth=1.0, label='u (east)')
    ax.plot(lags, res['rho_v'], color=color, linewidth=1.0, label='v (north)',
            linestyle='--')
    ax.axhline(0, color='k', linewidth=0.5)

    # Mark zero crossings
    for k_star, ls in [(res['k_star_u'], '-'), (res['k_star_v'], '--')]:
        ax.axvline(k_star * res['dt'] / 3600,
                   color=color, linestyle=ls, linewidth=0.8, alpha=0.6)

    T_u_hr = res['T_ET_u'] / 3600
    T_v_hr = res['T_ET_v'] / 3600
    ax.text(0.97, 0.95,
            f"$T_{{ET}}$ u = {T_u_hr:.1f} hr\n"
            f"$T_{{ET}}$ v = {T_v_hr:.1f} hr\n"
            f"N* u = {res['N_star_u']:.0f}\n"
            f"N* v = {res['N_star_v']:.0f}",
            transform=ax.transAxes,
            ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='gray', alpha=0.8))

    ax.set_title(moorings[name]['label'])
    ax.set_xlabel('Lag (hours)')
    ax.set_xlim(0, MAX_LAG_DAYS * 24)
    ax.set_ylim(-0.5, 1.05)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Autocorrelation')
axes[0].legend(fontsize=9)
plt.suptitle('Autocorrelation — full record (trapezoidal $T_{ET}$)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Full record table
rows = []
for name, res in timescale_results.items():
    for comp, T_ET, k_star, N_star, mean, se, ci in [
        ('u', res['T_ET_u'], res['k_star_u'], res['N_star_u'],
              res['mean_u'],  res['se_u'],     res['ci_u']),
        ('v', res['T_ET_v'], res['k_star_v'], res['N_star_v'],
              res['mean_v'],  res['se_v'],     res['ci_v']),
    ]:
        rows.append({
            'Mooring':       name,
            'Component':     comp,
            'T_ET (hr)':     round(T_ET / 3600, 2),
            'k* (lags)':     k_star,
            'N*':            round(N_star, 1),
            'Mean (cm/s)':   round(mean, 3),
            'SE (cm/s)':     round(se, 4),
            'CI lower':      round(ci[0], 3),
            'CI upper':      round(ci[1], 3),
        })

print("Full record")
display(pd.DataFrame(rows).set_index(['Mooring', 'Component']))

# Seasonal table
rows = []
for name in mooring_order:
    for season in season_order:
        if season not in timescale_seasonal[name]:
            continue
        res = timescale_seasonal[name][season]
        for comp, T_ET, k_star, N_star, mean, se, ci in [
            ('u', res['T_ET_u'], res['k_star_u'], res['N_star_u'],
                  res['mean_u'],  res['se_u'],     res['ci_u']),
            ('v', res['T_ET_v'], res['k_star_v'], res['N_star_v'],
                  res['mean_v'],  res['se_v'],     res['ci_v']),
        ]:
            rows.append({
                'Mooring':     name,
                'Season':      season,
                'Component':   comp,
                'N':           res['N'],
                'T_ET (hr)':   round(T_ET / 3600, 2),
                'k* (lags)':   k_star,
                'N*':          round(N_star, 1),
                'Mean (cm/s)': round(mean, 3),
                'SE (cm/s)':   round(se, 4),
                'CI lower':    round(ci[0], 3),
                'CI upper':    round(ci[1], 3),
            })

print("\nSeasonal")
display(pd.DataFrame(rows).set_index(['Mooring', 'Season', 'Component']))

### Rotate Mooring Velocities and estimate Integral timescale for along-trough flow 

In [ ]:
# def rotate_velocities(u, v, theta_deg):
#     """
#     Rotate (u=East, v=North) to (along-trough, across-trough).
    
#     Sign convention: positive v_along points from mooring 1 toward mooring 2.
#     Flip sign afterward if that's up-trough rather than toward shelf break.
    
#     Parameters
#     ----------
#     u, v      : array-like, eastward and northward velocity (m/s)
#     theta_deg : float, trough axis angle CCW from East
    
#     Returns
#     -------
#     v_along, v_across : rotated velocity components (m/s)
#     """
#     theta = np.deg2rad(theta_deg)
#     v_along  =  u * np.cos(theta) - v * np.sin(theta) # u_prime, positive towards shelf break 
#     v_across = u * np.sin(theta) + v * np.cos(theta) # v_prime, positive towards land (NW direction)
#     return v_along, v_across

def rotate_velocities(u, v, theta_deg):
    """
    Rotate (u=East, v=North) to (along-trough, across-trough).
    
    Sign convention: positive v_along points from mooring 1 toward mooring 2.
    Flip sign afterward if that's up-trough rather than toward shelf break.
    
    Parameters
    ----------
    u, v      : array-like, eastward and northward velocity (m/s)
    theta_deg : float, trough axis angle CCW from East
    
    Returns
    -------
    v_along, v_across : rotated velocity components (m/s)
    """
    theta = np.deg2rad(theta_deg)
    v_along  =  u * np.cos(theta) + v * np.sin(theta) # u_prime, positive towards shelf break 
    v_across = -u * np.sin(theta) + v * np.cos(theta) # v_prime, positive towards land (NW direction)
    return v_along, v_across


In [ ]:
# calculated in Transport_Calculation notebook, using same value here 
theta_deg = 34.5687 

In [ ]:
DITD_v_along, DITD_v_across = rotate_velocities(DITD_vel.u, DITD_vel.v, theta_deg)
DITD_vel['u_prime'] = ('time',DITD_v_along.data)
DITD_vel['v_prime'] = ('time',DITD_v_across.data)

In [ ]:
TNBD_v_along, TNBD_v_across = rotate_velocities(TNBD_vel.u, TNBD_vel.v, theta_deg)
TNBD_vel['u_prime'] = ('time',TNBD_v_along.data)
TNBD_vel['v_prime'] = ('time',TNBD_v_across.data)

In [ ]:
D_v_along, D_v_across = rotate_velocities(D_vel.u, D_vel.v, theta_deg)
D_vel['u_prime'] = ('time',D_v_along.data)
D_vel['v_prime'] = ('time',D_v_across.data)

In [ ]:

# Mooring registry — update fs once you confirm sampling intervals
# from ds.time.diff('time').median()
moorings = {
    'TNBD': {'ds': TNBD_vel, 'fs': 1/900,  'color': 'steelblue', 'label': 'TNBD'},
    'DITD': {'ds': DITD_vel, 'fs': 1/600,  'color': 'tomato',    'label': 'DITD'},
    'D':    {'ds': D_vel,    'fs': 1/1800,  'color': 'seagreen',  'label': 'D'},
}

In [ ]:
from TNB_tools.Integral_Timescale import (
    check_gaps,
    autocorrelation,
    integral_timescale_trapezoidal,
    one_sided_integral_timescale,
    effective_dof,
    standard_error_saunders,
    confidence_interval_mean,
)

MAX_LAG_DAYS = 70

timescale_results = {}

for name, m in moorings.items():
    ds = m['ds']
    dt = 1 / m['fs']
    u  = ds['u_prime'].values
    v  = ds['v_prime'].values
    N  = len(u)
    T_record = N * dt   # seconds

    nlags = int(MAX_LAG_DAYS * 86400 / dt)

    rho_u = autocorrelation(u, nlags=nlags)
    rho_v = autocorrelation(v, nlags=nlags)

    # Trapezoidal one-sided integral timescale (E&T convention)
    T_ET_u, k_star_u = integral_timescale_trapezoidal(rho_u, dt)
    T_ET_v, k_star_v = integral_timescale_trapezoidal(rho_v, dt)

    # Standard error on the mean (Saunders 1987)
    # Note: standard_error_saunders expects one-sided T_L
    # integral_timescale_trapezoidal already returns one-sided T, so use directly
    sigma_u   = u.std(ddof=1)
    sigma_v   = v.std(ddof=1)
    se_u      = standard_error_saunders(sigma_u, T_ET_u, T_record)
    se_v      = standard_error_saunders(sigma_v, T_ET_v, T_record)

    # N* for reference
    N_star_u  = effective_dof(N, dt, T_ET_u * 2)   # effective_dof expects two-sided T_int
    N_star_v  = effective_dof(N, dt, T_ET_v * 2)

    mean_u    = u.mean()
    mean_v    = v.mean()

    # 95% CI using normal approximation (Saunders style)
    from scipy.stats import norm
    z95 = norm.ppf(0.975)

    timescale_results[name] = {
        'lags_hours': np.arange(nlags + 1) * dt / 3600,
        'rho_u':      rho_u,
        'rho_v':      rho_v,
        'T_ET_u':     T_ET_u,
        'T_ET_v':     T_ET_v,
        'k_star_u':   k_star_u,
        'k_star_v':   k_star_v,
        'N_star_u':   N_star_u,
        'N_star_v':   N_star_v,
        'se_u':       se_u,
        'se_v':       se_v,
        'mean_u':     mean_u,
        'mean_v':     mean_v,
        'ci_u':       (mean_u - z95 * se_u, mean_u + z95 * se_u),
        'ci_v':       (mean_v - z95 * se_v, mean_v + z95 * se_v),
        'dt':         dt,
        'T_record_days': T_record / 86400,
    }

    print(f"\n{'='*45}")
    print(f"Mooring: {name}  (dt={dt:.0f}s, N={N}, "
          f"T_record={T_record/86400:.1f} days)")
    print(f"{'='*45}")
    for comp, T_ET, k_star, N_star, mean, se, ci in [
        ('u', T_ET_u, k_star_u, N_star_u, mean_u, se_u, timescale_results[name]['ci_u']),
        ('v', T_ET_v, k_star_v, N_star_v, mean_v, se_v, timescale_results[name]['ci_v']),
    ]:
        print(f"  {comp}: T_ET={T_ET/3600:.2f} hr  k*={k_star}  "
              f"N*={N_star:.0f}  mean={mean:.3f}  "
              f"SE={se:.4f}  95%CI=({ci[0]:.3f}, {ci[1]:.3f})")

In [ ]:
seasons = {
    'MAM': [3, 4, 5],
    'JJA': [6, 7, 8],
    'SON': [9, 10, 11],
    'DJF': [12],
}
season_notes = {
    'MAM': 'Mar–May',
    'JJA': 'Jun–Aug',
    'SON': 'Sep–Nov',
    'DJF': 'Dec only',
}
season_order  = ['MAM', 'JJA', 'SON', 'DJF']
mooring_order = list(moorings.keys())

timescale_seasonal = {name: {} for name in mooring_order}

for name in mooring_order:
    ds = moorings[name]['ds']
    dt = 1 / moorings[name]['fs']

    print(f"\n{'='*45}")
    print(f"Mooring: {name}")
    print(f"{'='*45}")

    for season, months in seasons.items():
        seasonal  = ds.sel(time=ds.time.dt.month.isin(months))
        u         = seasonal['u_prime'].values
        v         = seasonal['v_prime'].values
        N         = len(u)
        T_record  = N * dt

        # Need enough lags to find zero crossing but can't exceed record
        nlags = min(int(MAX_LAG_DAYS * 86400 / dt), N // 2)

        if N < 2 * nlags:
            print(f"  {season}: only {N} samples, skipping")
            continue

        if np.any(np.isnan(u)) or np.any(np.isnan(v)):
            print(f"  {season}: NaNs present, skipping")
            continue

        try:
            rho_u = autocorrelation(u, nlags=nlags)
            rho_v = autocorrelation(v, nlags=nlags)

            T_ET_u, k_star_u = integral_timescale_trapezoidal(rho_u, dt)
            T_ET_v, k_star_v = integral_timescale_trapezoidal(rho_v, dt)
        except ValueError as e:
            print(f"  {season}: {e}")
            continue

        sigma_u  = u.std(ddof=1)
        sigma_v  = v.std(ddof=1)
        se_u     = standard_error_saunders(sigma_u, T_ET_u, T_record)
        se_v     = standard_error_saunders(sigma_v, T_ET_v, T_record)
        N_star_u = effective_dof(N, dt, T_ET_u * 2)
        N_star_v = effective_dof(N, dt, T_ET_v * 2)
        mean_u   = u.mean()
        mean_v   = v.mean()

        timescale_seasonal[name][season] = {
            'T_ET_u':    T_ET_u,
            'T_ET_v':    T_ET_v,
            'k_star_u':  k_star_u,
            'k_star_v':  k_star_v,
            'N_star_u':  N_star_u,
            'N_star_v':  N_star_v,
            'se_u':      se_u,
            'se_v':      se_v,
            'mean_u':    mean_u,
            'mean_v':    mean_v,
            'ci_u':      (mean_u - z95 * se_u, mean_u + z95 * se_u),
            'ci_v':      (mean_v - z95 * se_v, mean_v + z95 * se_v),
            'N':         N,
            'T_record_days': T_record / 86400,
        }

        print(f"  {season} ({season_notes[season]}, N={N}):")
        for comp, T_ET, k_star, N_star, mean, se in [
            ('u', T_ET_u, k_star_u, N_star_u, mean_u, se_u),
            ('v', T_ET_v, k_star_v, N_star_v, mean_v, se_v),
        ]:
            print(f"    {comp}: T_ET={T_ET/3600:.2f} hr  k*={k_star}  "
                  f"N*={N_star:.0f}  mean={mean:.3f}  SE={se:.4f}")

In [ ]:
fig, axes = plt.subplots(len(mooring_order), len(season_order),
                          figsize=(16, 10), sharex=True, sharey=True)

for i, name in enumerate(mooring_order):
    ds = moorings[name]['ds']
    dt = 1 / moorings[name]['fs']

    for j, season in enumerate(season_order):
        ax = axes[i, j]
        months = seasons[season]
        seasonal = ds.sel(time=ds.time.dt.month.isin(months))
        u = seasonal['u_prime'].values
        v = seasonal['v_prime'].values
        N = len(u)

        nlags = min(int(MAX_LAG_DAYS * 86400 / dt), N // 2)

        # same skip logic as the timescale-computation cell
        if N < 2 * nlags or np.any(np.isnan(u)) or np.any(np.isnan(v)):
            ax.text(0.5, 0.5, 'insufficient data', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color='gray')
            ax.set_title(f"{name} — {season}", fontsize=10)
            continue

        rho_u = autocorrelation(u, nlags=nlags)
        rho_v = autocorrelation(v, nlags=nlags)
        lag_days = np.arange(len(rho_u)) * dt / 86400

        # pull the already-computed k_star / T_ET from the stored results
        # (recomputed rho here just for plotting, so these should match)
        res = timescale_seasonal[name].get(season)

        ax.plot(lag_days, rho_u, color='tab:blue', lw=1, label='u')
        ax.plot(lag_days, rho_v, color='tab:orange', lw=1, label='v')
        ax.axhline(0, color='k', lw=0.7)

        if res is not None:
            k_star_u_days = res['k_star_u'] * dt / 86400
            k_star_v_days = res['k_star_v'] * dt / 86400
            ax.axvline(k_star_u_days, color='tab:blue', lw=0.8, ls='--', alpha=0.6)
            ax.axvline(k_star_v_days, color='tab:orange', lw=0.8, ls='--', alpha=0.6)

        ax.set_title(f"{name} — {season}", fontsize=10)

        if i == len(mooring_order) - 1:
            ax.set_xlabel('Lag (days)')
        if j == 0:
            ax.set_ylabel(r'$\rho$')

# single shared legend instead of repeating per-panel
handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper right', ncol=2, frameon=False)

fig.suptitle('Seasonal autocorrelation by mooring (dashed = zero-crossing cutoff)', y=1.02)
plt.tight_layout()
plt.show()

### Repeat Analysis on Rotated Velocity but use low pass filtered data to remove tidal effects 

In [ ]:
from TNB_tools.tides.tidal_decomposition import decompose_tidal_dataset

In [ ]:
# using functions 
TNBD_lowpass_ds, TNBD_highpass_ds = decompose_tidal_dataset(
    TNBD_vel, var_names=["u_prime", "v_prime"], window_days=2, sampling_hours=0.25
)


DITD_lowpass_ds, DITD_highpass_ds = decompose_tidal_dataset(
    DITD_vel, var_names=["u_prime", "v_prime"], window_days=2, sampling_hours=0.25
)

D_lowpass_ds, D_highpass_ds = decompose_tidal_dataset(
    D_vel, var_names=["u_prime", "v_prime"], window_days=2, sampling_hours=0.25
)

In [ ]:
TNBD_lowpass_ds = TNBD_lowpass_ds.dropna(dim='time')
DITD_lowpass_ds = DITD_lowpass_ds.dropna(dim='time')
D_lowpass_ds = D_lowpass_ds.dropna(dim='time')

In [ ]:
# Mooring registry — update fs once you confirm sampling intervals
# from ds.time.diff('time').median()
moorings = {
    'TNBD': {'ds': TNBD_lowpass_ds, 'fs': 1/900,  'color': 'steelblue', 'label': 'TNBD'},
    'DITD': {'ds': DITD_lowpass_ds, 'fs': 1/600,  'color': 'tomato',    'label': 'DITD'},
    'D':    {'ds': D_lowpass_ds,    'fs': 1/1800,  'color': 'seagreen',  'label': 'D'},
}

In [ ]:
from TNB_tools.Integral_Timescale import (
    check_gaps,
    autocorrelation,
    integral_timescale_trapezoidal,
    one_sided_integral_timescale,
    effective_dof,
    standard_error_saunders,
    confidence_interval_mean,
)

MAX_LAG_DAYS = 70

timescale_results = {}

for name, m in moorings.items():
    ds = m['ds']
    dt = 1 / m['fs']
    u  = ds['u_prime'].values
    v  = ds['v_prime'].values
    N  = len(u)
    T_record = N * dt   # seconds

    nlags = int(MAX_LAG_DAYS * 86400 / dt)

    rho_u = autocorrelation(u, nlags=nlags)
    rho_v = autocorrelation(v, nlags=nlags)

    # Trapezoidal one-sided integral timescale (E&T convention)
    T_ET_u, k_star_u = integral_timescale_trapezoidal(rho_u, dt)
    T_ET_v, k_star_v = integral_timescale_trapezoidal(rho_v, dt)

    # Standard error on the mean (Saunders 1987)
    # Note: standard_error_saunders expects one-sided T_L
    # integral_timescale_trapezoidal already returns one-sided T, so use directly
    sigma_u   = u.std(ddof=1)
    sigma_v   = v.std(ddof=1)
    se_u      = standard_error_saunders(sigma_u, T_ET_u, T_record)
    se_v      = standard_error_saunders(sigma_v, T_ET_v, T_record)

    # N* for reference
    N_star_u  = effective_dof(N, dt, T_ET_u * 2)   # effective_dof expects two-sided T_int
    N_star_v  = effective_dof(N, dt, T_ET_v * 2)

    mean_u    = u.mean()
    mean_v    = v.mean()

    # 95% CI using normal approximation (Saunders style)
    from scipy.stats import norm
    z95 = norm.ppf(0.975)

    timescale_results[name] = {
        'lags_hours': np.arange(nlags + 1) * dt / 3600,
        'rho_u':      rho_u,
        'rho_v':      rho_v,
        'T_ET_u':     T_ET_u,
        'T_ET_v':     T_ET_v,
        'k_star_u':   k_star_u,
        'k_star_v':   k_star_v,
        'N_star_u':   N_star_u,
        'N_star_v':   N_star_v,
        'se_u':       se_u,
        'se_v':       se_v,
        'mean_u':     mean_u,
        'mean_v':     mean_v,
        'ci_u':       (mean_u - z95 * se_u, mean_u + z95 * se_u),
        'ci_v':       (mean_v - z95 * se_v, mean_v + z95 * se_v),
        'dt':         dt,
        'T_record_days': T_record / 86400,
    }

    print(f"\n{'='*45}")
    print(f"Mooring: {name}  (dt={dt:.0f}s, N={N}, "
          f"T_record={T_record/86400:.1f} days)")
    print(f"{'='*45}")
    for comp, T_ET, k_star, N_star, mean, se, ci in [
        ('u', T_ET_u, k_star_u, N_star_u, mean_u, se_u, timescale_results[name]['ci_u']),
        ('v', T_ET_v, k_star_v, N_star_v, mean_v, se_v, timescale_results[name]['ci_v']),
    ]:
        print(f"  {comp}: T_ET={T_ET/3600:.2f} hr  k*={k_star}  "
              f"N*={N_star:.0f}  mean={mean:.3f}  "
              f"SE={se:.4f}  95%CI=({ci[0]:.3f}, {ci[1]:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (name, res) in zip(axes, timescale_results.items()):
    color = moorings[name]['color']
    lags  = res['lags_hours']

    ax.plot(lags, res['rho_u'], color=color, linewidth=1.0, label='u (along-trough)')
    ax.plot(lags, res['rho_v'], color=color, linewidth=1.0, label='v (across-trough)',
            linestyle='--')
    ax.axhline(0, color='k', linewidth=0.5)

    # Mark zero crossings
    for k_star, ls in [(res['k_star_u'], '-'), (res['k_star_v'], '--')]:
        ax.axvline(k_star * res['dt'] / 3600,
                   color=color, linestyle=ls, linewidth=0.8, alpha=0.6)

    T_u_hr = res['T_ET_u'] / 3600
    T_v_hr = res['T_ET_v'] / 3600
    # ax.text(0.97, 0.95,
    #         f"$T_{{ET}}$ u = {T_u_hr:.1f} hr\n"
    #         f"$T_{{ET}}$ v = {T_v_hr:.1f} hr\n"
    #         f"N* u = {res['N_star_u']:.0f}\n"
    #         f"N* v = {res['N_star_v']:.0f}",
    #         transform=ax.transAxes,
    #         ha='right', va='top', fontsize=8,
    #         bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
    #                   edgecolor='gray', alpha=0.8))

    ax.set_title(moorings[name]['label'])
    ax.set_xlabel('Lag (hours)')
    ax.set_xlim(0, MAX_LAG_DAYS * 24)
    ax.set_ylim(-0.5, 1.05)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Autocorrelation')
axes[0].legend(fontsize=9)
# plt.suptitle('Autocorrelation — full record (trapezoidal $T_{ET}$)', y=1.02)
plt.tight_layout()
# plt.savefig('FullRecord_IntegralTimescale.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
seasons = {
    'MAM': [3, 4, 5],
    'JJA': [6, 7, 8],
    'SON': [9, 10, 11],
    'DJF': [12],
}
season_notes = {
    'MAM': 'Mar–May',
    'JJA': 'Jun–Aug',
    'SON': 'Sep–Nov',
    'DJF': 'Dec only',
}
season_order  = ['MAM', 'JJA', 'SON', 'DJF']
mooring_order = list(moorings.keys())

timescale_seasonal = {name: {} for name in mooring_order}

for name in mooring_order:
    ds = moorings[name]['ds']
    dt = 1 / moorings[name]['fs']

    print(f"\n{'='*45}")
    print(f"Mooring: {name}")
    print(f"{'='*45}")

    for season, months in seasons.items():
        seasonal  = ds.sel(time=ds.time.dt.month.isin(months))
        u         = seasonal['u_prime'].values
        v         = seasonal['v_prime'].values
        N         = len(u)
        T_record  = N * dt

        # Need enough lags to find zero crossing but can't exceed record
        nlags = min(int(MAX_LAG_DAYS * 86400 / dt), N // 2)

        if N < 2 * nlags:
            print(f"  {season}: only {N} samples, skipping")
            continue

        if np.any(np.isnan(u)) or np.any(np.isnan(v)):
            print(f"  {season}: NaNs present, skipping")
            continue

        try:
            rho_u = autocorrelation(u, nlags=nlags)
            rho_v = autocorrelation(v, nlags=nlags)

            T_ET_u, k_star_u = integral_timescale_trapezoidal(rho_u, dt)
            T_ET_v, k_star_v = integral_timescale_trapezoidal(rho_v, dt)
        except ValueError as e:
            print(f"  {season}: {e}")
            continue

        sigma_u  = u.std(ddof=1)
        sigma_v  = v.std(ddof=1)
        se_u     = standard_error_saunders(sigma_u, T_ET_u, T_record)
        se_v     = standard_error_saunders(sigma_v, T_ET_v, T_record)
        N_star_u = effective_dof(N, dt, T_ET_u * 2)
        N_star_v = effective_dof(N, dt, T_ET_v * 2)
        mean_u   = u.mean()
        mean_v   = v.mean()

        timescale_seasonal[name][season] = {
            'T_ET_u':    T_ET_u,
            'T_ET_v':    T_ET_v,
            'k_star_u':  k_star_u,
            'k_star_v':  k_star_v,
            'N_star_u':  N_star_u,
            'N_star_v':  N_star_v,
            'se_u':      se_u,
            'se_v':      se_v,
            'mean_u':    mean_u,
            'mean_v':    mean_v,
            'ci_u':      (mean_u - z95 * se_u, mean_u + z95 * se_u),
            'ci_v':      (mean_v - z95 * se_v, mean_v + z95 * se_v),
            'N':         N,
            'T_record_days': T_record / 86400,
        }

        print(f"  {season} ({season_notes[season]}, N={N}):")
        for comp, T_ET, k_star, N_star, mean, se in [
            ('u', T_ET_u, k_star_u, N_star_u, mean_u, se_u),
            ('v', T_ET_v, k_star_v, N_star_v, mean_v, se_v),
        ]:
            print(f"    {comp}: T_ET={T_ET/3600:.2f} hr  k*={k_star}  "
                  f"N*={N_star:.0f}  mean={mean:.3f}  SE={se:.4f}")

In [ ]:
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,      # x/y axis labels
    'axes.titlesize': 12,      # panel titles (a), (b), etc.
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 13,    # overall figure suptitle, if used
})

In [ ]:
fig, axes = plt.subplots(len(mooring_order), len(season_order),
                          figsize=(16, 10), sharex=True, sharey=True)

for i, name in enumerate(mooring_order):
    ds = moorings[name]['ds']
    dt = 1 / moorings[name]['fs']

    for j, season in enumerate(season_order):
        ax = axes[i, j]
        months = seasons[season]
        seasonal = ds.sel(time=ds.time.dt.month.isin(months))
        u = seasonal['u_prime'].values
        v = seasonal['v_prime'].values
        N = len(u)

        nlags = min(int(MAX_LAG_DAYS * 86400 / dt), N // 2)

        # same skip logic as the timescale-computation cell
        if N < 2 * nlags or np.any(np.isnan(u)) or np.any(np.isnan(v)):
            ax.text(0.5, 0.5, 'insufficient data', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color='gray')
            ax.set_title(f"{name} — {season}", fontsize=10)
            continue

        rho_u = autocorrelation(u, nlags=nlags)
        rho_v = autocorrelation(v, nlags=nlags)
        lag_days = np.arange(len(rho_u)) * dt / 86400

        # pull the already-computed k_star / T_ET from the stored results
        # (recomputed rho here just for plotting, so these should match)
        res = timescale_seasonal[name].get(season)

        ax.plot(lag_days, rho_u, color='tab:blue', lw=1, label='Along-Trough')
        ax.plot(lag_days, rho_v, color='tab:orange', lw=1, label='Across-Trough')
        ax.axhline(0, color='k', lw=0.7)
        

        if res is not None:
            k_star_u_days = res['k_star_u'] * dt / 86400
            k_star_v_days = res['k_star_v'] * dt / 86400
            ax.axvline(k_star_u_days, color='tab:blue', lw=0.8, ls='--', alpha=0.6)
            ax.axvline(k_star_v_days, color='tab:orange', lw=0.8, ls='--', alpha=0.6)

        ax.set_title(f"{name} — {season}", fontsize=10)

        if i == len(mooring_order) - 1:
            ax.set_xlabel('Lag (days)')
        if j == 0:
            ax.set_ylabel(r'Autocorrelation')

# single shared legend instead of repeating per-panel
handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, ncol=1, frameon=False, loc='upper right',bbox_to_anchor=(0.98, 0.97))

# fig.suptitle('Seasonal autocorrelation by mooring (dashed = zero-crossing cutoff)', y=1.02)
plt.tight_layout()
# plt.savefig('Seasonal_IntegralTimescale.png', dpi=300, bbox_inches='tight')
plt.show()

### Integral Timescale and Standard Error Sensitivity Analysis on Monthly Basis 

Goal: Understand how integral timescale and standard error estimates change for monthly means. Specifically, I am interested in understanding what is the difference in standard error when calculated using the full record integral timescale versus the monthly integral timescale. Hopefully these differences are small and then I can pick whichever method to use for getting standard error. 

In [ ]:
# load in data 

# LOAD IN INTERMEDIATE DATASETS DELL - 30 MIN AVERAGED DATA 
TNBD_ctd_top = xr.open_dataset('data/TNBD_ctd_top_30min.nc')
TNBD_ctd_bot = xr.open_dataset('data/TNBD_ctd_bot_30min.nc')
TNBD_vel_top = xr.open_dataset('data/TNBD_vel_top_30min.nc')
TNBD_vel_bot = xr.open_dataset('data/TNBD_vel_bot_30min.nc')

DITD_ctd_top = xr.open_dataset('data/DITD_ctd_top_30min.nc')
DITD_ctd_bot = xr.open_dataset('data/DITD_ctd_bot_30min.nc')
DITD_vel_top = xr.open_dataset('data/DITD_vel_top_30min.nc')
DITD_vel_bot = xr.open_dataset('data/DITD_vel_bot_30min.nc')      # sensor cut out in june 2018

D_ctd_top = xr.open_dataset('data/D_ctd_top_30min.nc')
D_ctd_mid = xr.open_dataset('data/D_ctd_mid_30min.nc')
# D_ctd_bot = D_ctd_bot.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out in 2017

# D_vel_top = D_vel_top.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out end of 2017
D_vel_mid = xr.open_dataset('data/D_vel_mid_30min.nc')
D_vel_bot = xr.open_dataset('data/D_vel_bot_30min.nc')

In [ ]:
TNBD_vel_top

In [ ]:
# rotate 


In [ ]:
DITD_v_along, DITD_v_across = rotate_velocities(DITD_vel_top.u, DITD_vel_top.v, theta_deg)
DITD_vel_top['u_prime'] = ('time',DITD_v_along.data)
DITD_vel_top['v_prime'] = ('time',DITD_v_across.data)

In [ ]:
TNBD_v_along, TNBD_v_across = rotate_velocities(TNBD_vel_bot.u, TNBD_vel_bot.v, theta_deg)
TNBD_vel_bot['u_prime'] = ('time',TNBD_v_along.data)
TNBD_vel_bot['v_prime'] = ('time',TNBD_v_across.data)

In [ ]:
D_v_along, D_v_across = rotate_velocities(D_vel_bot.u, D_vel_bot.v, theta_deg)
D_vel_bot['u_prime'] = ('time',D_v_along.data)
D_vel_bot['v_prime'] = ('time',D_v_across.data)

In [ ]:
# using functions 
TNBD_lowpass_ds, TNBD_highpass_ds = decompose_tidal_dataset(
    TNBD_vel_bot, var_names=["u_prime", "v_prime"], window_days=3, sampling_hours=0.5
)


DITD_lowpass_ds, DITD_highpass_ds = decompose_tidal_dataset(
    DITD_vel_top, var_names=["u_prime", "v_prime"], window_days=3, sampling_hours=0.5
)

D_lowpass_ds, D_highpass_ds = decompose_tidal_dataset(
    D_vel_bot, var_names=["u_prime", "v_prime"], window_days=3, sampling_hours=0.5
)

In [ ]:
TNBD_lowpass_ds

In [ ]:
# import numpy as np
# import pandas as pd
# import xarray as xr
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# import sys
# sys.path.append('path/to/your/TNB_tools')  # adjust to your repo path
# import velocity_timescale as vt

# ── MOORING DATASETS ──────────────────────────────────────────────────────────
# Each should be an xarray Dataset with 'u_prime' variable and 'time' coord
# Replace with your actual dataset names
MOORINGS = {
    'DITD': {'ds': DITD_lowpass_ds, 'color': 'tomato',    'dt_hours': 0.5},
    'TNBD': {'ds': TNBD_lowpass_ds, 'color': 'steelblue', 'dt_hours': 0.5},
    'D':    {'ds': D_lowpass_ds,    'color': 'seagreen',  'dt_hours': 0.5},
}

# ── HELPER: clean a time series for velocity_timescale.py ────────────────────
def prepare_series(x):
    """Interpolate NaNs so velocity_timescale functions don't error."""
    x = np.asarray(x, dtype=float)
    t = np.arange(len(x))
    finite = np.isfinite(x)
    if finite.sum() < 10:
        return None
    return np.interp(t, t[finite], x[finite])


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — FULL RECORD INTEGRAL TIMESCALE
# ══════════════════════════════════════════════════════════════════════════════

full_record_results = {}

for name, meta in MOORINGS.items():
    ds       = meta['ds']
    dt_hours = meta['dt_hours']
    dt_s     = dt_hours * 3600.0

    x = prepare_series(ds['u_prime'].values)
    if x is None:
        print(f"{name}: insufficient data"); continue

    # Autocorrelation — search up to 10 days for zero crossing
    nlags_10d = int(20 * 24 / dt_hours)
    rho       = its.autocorrelation(x, nlags=nlags_10d)

    # Two-sided T_int then convert to one-sided T_L (Saunders convention)
    T_int, k_star = its.integral_timescale(rho, dt=dt_s)
    T_L           = its.one_sided_integral_timescale(T_int)   # T_int / 2

    # Full record standard error (Saunders 1987)
    T_record = len(x) * dt_s
    sigma    = np.std(x, ddof=1)
    SE_full  = its.standard_error_saunders(sigma, T_L, T_record)
    N_star   = its.effective_dof(len(x), dt_s, T_int)

    full_record_results[name] = {
        'T_int_s':   T_int,
        'T_int_h':   T_int / 3600,
        'T_int_d':   T_int / 86400,
        'T_L_s':     T_L,
        'T_L_h':     T_L / 3600,
        'T_L_d':     T_L / 86400,
        'k_star':    k_star,
        'N_star':    N_star,
        'sigma':     sigma,
        'SE_full':   SE_full,
        'rho':       rho,
        'N':         len(x),
        'dt_s':      dt_s,
    }

    print(f"\n{'='*55}")
    print(f"{name} — full record")
    print(f"  N:              {len(x)}")
    print(f"  T_int (2-sided): {T_int/3600:.2f} h  ({T_int/86400:.2f} d)")
    print(f"  T_L   (1-sided): {T_L/3600:.2f} h  ({T_L/86400:.2f} d)")
    print(f"  k_star:          {k_star} lags")
    print(f"  N*:              {N_star:.1f}")
    print(f"  sigma:           {sigma:.4f} m/s")
    print(f"  SE (full):       {SE_full:.5f} m/s")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — MONTHLY ESTIMATES
# ══════════════════════════════════════════════════════════════════════════════

monthly_results = {}

for name, meta in MOORINGS.items():
    if name not in full_record_results:
        continue

    ds       = meta['ds']
    dt_hours = meta['dt_hours']
    dt_s     = dt_hours * 3600.0
    fr       = full_record_results[name]   # full record results shorthand

    time     = pd.DatetimeIndex(ds['time'].values)
    x_all    = ds['u_prime'].values.astype(float)
    records  = []

    for month in sorted(time.month.unique()):
        mask    = time.month == month
        x_month = x_all[mask]
        label   = pd.Timestamp(f'2018-{month:02d}-01').strftime('%b')
        n_valid = np.sum(np.isfinite(x_month))

        if n_valid < 20:
            print(f"  {name} {label}: skipping ({n_valid} valid pts)")
            continue

        x_clean  = prepare_series(x_month)
        sigma_mo = np.std(x_clean, ddof=1)
        T_record_mo = len(x_clean) * dt_s

        # ── SE using FULL-RECORD T_L, monthly sigma and record length ─────────
        SE_fullTL = its.standard_error_saunders(
            sigma_mo, fr['T_L_s'], T_record_mo)
        N_star_fullTL = its.effective_dof(len(x_clean), dt_s, fr['T_int_s'])

        # ── Monthly T_L ───────────────────────────────────────────────────────
        try:
            nlags_mo = min(int(20 * 24 / dt_hours), len(x_clean) // 2)
            rho_mo   = its.autocorrelation(x_clean, nlags=nlags_mo)
            T_int_mo, k_star_mo = its.integral_timescale(rho_mo, dt=dt_s)
            T_L_mo   = its.one_sided_integral_timescale(T_int_mo)

            # Guard: T_L longer than record makes no physical sense
            if T_int_mo >= T_record_mo:
                print(f"  {name} {label}: T_int >= T_record — "
                      f"capping at full-record T_L")
                T_int_mo = fr['T_int_s']
                T_L_mo   = fr['T_L_s']
                k_star_mo = fr['k_star']

            SE_monthlyTL  = its.standard_error_saunders(
                sigma_mo, T_L_mo, T_record_mo)
            N_star_monthly = its.effective_dof(
                len(x_clean), dt_s, T_int_mo)

        except ValueError as e:
            print(f"  {name} {label}: ACF issue ({e}) — using full T_L")
            T_int_mo      = fr['T_int_s']
            T_L_mo        = fr['T_L_s']
            k_star_mo     = fr['k_star']
            SE_monthlyTL  = SE_fullTL
            N_star_monthly = N_star_fullTL
            rho_mo        = None

        records.append({
            'month':          label,
            'month_num':      month,
            'n':              len(x_clean),
            'sigma':          sigma_mo,
            'T_int_mo_h':     T_int_mo / 3600,
            'T_int_mo_d':     T_int_mo / 86400,
            'T_L_mo_h':       T_L_mo   / 3600,
            'T_L_mo_d':       T_L_mo   / 86400,
            'k_star_mo':      k_star_mo,
            'N_star_fullTL':  N_star_fullTL,
            'N_star_monthly': N_star_monthly,
            'SE_fullTL':      SE_fullTL,
            'SE_monthlyTL':   SE_monthlyTL,
            'rho_mo':         rho_mo,
            'dt_s':           dt_s,
        })

    monthly_results[name] = pd.DataFrame(records)


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — DATA TABLE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*80}")
print("SUMMARY TABLE — Monthly SE estimates")
print(f"{'='*80}")

for name, df in monthly_results.items():
    fr = full_record_results[name]
    print(f"\n{name}  |  Full-record T_L = {fr['T_L_d']:.2f} days  "
          f"|  Full-record SE = {fr['SE_full']:.5f} m/s")
    print(f"{'─'*75}")
    print(f"{'Month':<6} {'N':>6} {'sigma':>8} {'T_L_mo (d)':>12} "
          f"{'N* (full TL)':>14} {'SE (full TL)':>14} "
          f"{'N* (mo TL)':>12} {'SE (mo TL)':>12}")
    print(f"{'─'*75}")
    for _, row in df.iterrows():
        print(f"{row['month']:<6} {row['n']:>6} {row['sigma']:>8.4f} "
              f"{row['T_L_mo_d']:>12.2f} "
              f"{row['N_star_fullTL']:>14.1f} {row['SE_fullTL']:>14.5f} "
              f"{row['N_star_monthly']:>12.1f} {row['SE_monthlyTL']:>12.5f}")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — PLOTS
# ══════════════════════════════════════════════════════════════════════════════

for name, df in monthly_results.items():
    fr    = full_record_results[name]
    color = MOORINGS[name]['color']
    mons  = df['month'].values
    x     = np.arange(len(mons))

    fig = plt.figure(figsize=(16, 14))
    gs  = gridspec.GridSpec(3, 2, hspace=0.42, wspace=0.32,
                             figure=fig)

    # ── A: Full record ACF ────────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    rho_full = fr['rho']
    dt_h     = fr['dt_s'] / 3600
    lags_d   = np.arange(len(rho_full)) * dt_h / 24
    k_star   = fr['k_star']

    ax.plot(lags_d, rho_full, color=color, lw=2, label='Full record ACF')
    ax.fill_between(lags_d[:k_star+1], 0, rho_full[:k_star+1],
                    alpha=0.18, color=color,
                    label=f'T_L = {fr["T_L_d"]:.2f} d')
    ax.axvline(lags_d[k_star], color=color, lw=1.5, ls='--', alpha=0.7,
               label=f'Zero crossing = {lags_d[k_star]:.1f} d')
    ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
    ax.set_xlabel('Lag (days)', fontsize=11)
    ax.set_ylabel('Autocorrelation', fontsize=11)
    ax.set_ylim(-0.5, 1.05)
    ax.set_title(f'[A] Full record ACF — {name}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── B: Monthly ACFs overlaid ──────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 1])
    cmap   = plt.cm.twilight
    n_mo   = len(df)
    colors = [cmap(i / max(n_mo-1, 1)) for i in range(n_mo)]

    for (_, row), c in zip(df.iterrows(), colors):
        if row['rho_mo'] is not None:
            rho_mo = row['rho_mo']
            lags_mo = np.arange(len(rho_mo)) * (row['dt_s']/3600) / 24
            ax.plot(lags_mo, rho_mo, color=c, lw=1.5, alpha=0.85,
                    label=f"{row['month']} ({row['T_L_mo_d']:.1f}d)")

    # Full record ACF for reference
    ax.plot(lags_d, rho_full, color='k', lw=2.5, ls='--',
            label='Full record', zorder=5)
    ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
    ax.set_xlabel('Lag (days)', fontsize=11)
    ax.set_ylabel('Autocorrelation', fontsize=11)
    ax.set_ylim(-0.5, 1.05)
    ax.set_title(f'[B] Monthly ACFs — {name}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right', ncol=2, framealpha=0.8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── C: Monthly T_L vs full-record T_L ────────────────────────────────────
    ax = fig.add_subplot(gs[1, 0])
    ax.bar(x, df['T_L_mo_d'], color=color, alpha=0.7, label='Monthly T_L')
    ax.axhline(fr['T_L_d'], color='k', lw=2, ls='--',
               label=f'Full-record T_L = {fr["T_L_d"]:.2f} d')
    ax.set_xticks(x); ax.set_xticklabels(mons, fontsize=10)
    ax.set_ylabel('T_L (days)', fontsize=11)
    ax.set_title(f'[C] Monthly vs full-record T_L — {name}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── D: N_eff comparison ───────────────────────────────────────────────────
    ax = fig.add_subplot(gs[1, 1])
    ax.bar(x - 0.2, df['N_star_fullTL'],  width=0.38, color='k',
           alpha=0.5, label='N* (full-record T_L)')
    ax.bar(x + 0.2, df['N_star_monthly'], width=0.38, color=color,
           alpha=0.7, label='N* (monthly T_L)')
    ax.set_xticks(x); ax.set_xticklabels(mons, fontsize=10)
    ax.set_ylabel('N* (effective DOF)', fontsize=11)
    ax.set_title(f'[D] N* comparison — {name}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── E: SE comparison — the key figure ────────────────────────────────────
    ax = fig.add_subplot(gs[2, :])
    ax.bar(x - 0.2, df['SE_fullTL'],    width=0.38, color='k',
           alpha=0.5, label='SE — full-record T_L, monthly σ and N')
    ax.bar(x + 0.2, df['SE_monthlyTL'], width=0.38, color=color,
           alpha=0.7, label='SE — monthly T_L, monthly σ and N')
    ax.axhline(fr['SE_full'], color='k', lw=2, ls='--',
               label=f'SE full record = {fr["SE_full"]:.4f} m/s')
    ax.set_xticks(x); ax.set_xticklabels(mons, fontsize=10)
    ax.set_ylabel('Standard error (m/s)', fontsize=11)
    ax.set_title(f'[E] SE comparison — {name}\n'
                 f'If bars are similar → use full-record T_L for all months',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    fig.suptitle(f'Integral timescale and SE analysis — {name} u_prime',
                 fontsize=13, fontweight='bold')
    plt.savefig(f'timescale_analysis_{name}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Imports and settings
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from TNB_tools import Integral_Timescale as its

# Mooring datasets and metadata — adjust dataset names to match yours
MOORINGS = {
    'DITD': {'ds': DITD_lowpass_ds, 'color': 'tomato',    'dt_hours': 0.5},
    'TNBD': {'ds': TNBD_lowpass_ds, 'color': 'steelblue', 'dt_hours': 0.5},
    'D':    {'ds': D_lowpass_ds,    'color': 'seagreen',  'dt_hours': 0.5},
}

MAX_LAG_DAYS = 70   # search window for ACF zero crossing

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Helper
# ═══════════════════════════════════════════════════════════════════════════════

def prepare_series(x):
    """Interpolate NaNs so Integral_Timescale functions don't error."""
    x = np.asarray(x, dtype=float)
    t = np.arange(len(x))
    finite = np.isfinite(x)
    if finite.sum() < 10:
        return None
    return np.interp(t, t[finite], x[finite])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Full record integral timescale and SE
# ═══════════════════════════════════════════════════════════════════════════════

full_record_results = {}

for name, meta in MOORINGS.items():
    ds       = meta['ds']
    dt_hours = meta['dt_hours']
    dt_s     = dt_hours * 3600.0

    x = prepare_series(ds['u_prime'].values)
    if x is None:
        print(f"{name}: insufficient data")
        continue

    nlags_max = min(int(MAX_LAG_DAYS * 24 / dt_hours), len(x) // 2)
    rho       = its.autocorrelation(x, nlags=nlags_max)

    # Integral timescale with fallback if ACF never crosses zero
    try:
        T_int, k_star   = its.integral_timescale(rho, dt=dt_s)
        zero_cross_flag = False
    except ValueError:
        print(f"\n  WARNING [{name}]: ACF did not cross zero in "
              f"{MAX_LAG_DAYS} days — integrating positive area only.")
        positive = rho > 0
        lags_s   = np.arange(len(rho)) * dt_s
        T_int    = 2 * float(np.trapezoid(rho[positive], lags_s[positive]))
        k_star   = int(positive.sum())
        zero_cross_flag = True

    T_L      = its.one_sided_integral_timescale(T_int)
    T_record = len(x) * dt_s
    sigma    = np.std(x, ddof=1)
    SE_full  = its.standard_error_saunders(sigma, T_L, T_record)
    N_star   = its.effective_dof(len(x), dt_s, T_int)

    full_record_results[name] = {
        'T_int_s':         T_int,
        'T_int_h':         T_int / 3600,
        'T_int_d':         T_int / 86400,
        'T_L_s':           T_L,
        'T_L_h':           T_L / 3600,
        'T_L_d':           T_L / 86400,
        'k_star':          k_star,
        'N_star':          N_star,
        'sigma':           sigma,
        'SE_full':         SE_full,
        'rho':             rho,
        'N':               len(x),
        'dt_s':            dt_s,
        'zero_cross_flag': zero_cross_flag,
    }

    flag = "  ⚠ ACF fallback" if zero_cross_flag else ""
    print(f"\n{'='*52}")
    print(f"{name} — full record u_prime{flag}")
    print(f"  N:               {len(x)}")
    print(f"  T_int (2-sided): {T_int/3600:.2f} h  ({T_int/86400:.2f} d)")
    print(f"  T_L   (1-sided): {T_L/3600:.2f} h  ({T_L/86400:.2f} d)")
    print(f"  k_star:          {k_star} lags")
    print(f"  N*:              {N_star:.1f}")
    print(f"  sigma:           {sigma:.4f} cm/s")
    print(f"  SE (full):       {SE_full:.4f} cm/s")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Monthly integral timescales and SE
# ═══════════════════════════════════════════════════════════════════════════════

monthly_results = {}

for name, meta in MOORINGS.items():
    if name not in full_record_results:
        continue

    ds       = meta['ds']
    dt_hours = meta['dt_hours']
    dt_s     = dt_hours * 3600.0
    fr       = full_record_results[name]
    time     = pd.DatetimeIndex(ds['time'].values)
    x_all    = ds['u_prime'].values.astype(float)
    records  = []

    print(f"\n{'='*52}")
    print(f"{name} — monthly breakdown")
    print(f"  Full-record T_L = {fr['T_L_d']:.2f} days")
    print(f"{'─'*52}")

    for month in sorted(time.month.unique()):
        mask    = time.month == month
        x_month = x_all[mask]
        label   = pd.Timestamp(f'2018-{month:02d}-01').strftime('%b')
        n_valid = np.sum(np.isfinite(x_month))

        if n_valid < 20:
            print(f"  {label}: skipping ({n_valid} valid pts)")
            continue

        x_clean     = prepare_series(x_month)
        sigma_mo    = np.std(x_clean, ddof=1)
        T_record_mo = len(x_clean) * dt_s

        # ── SE using full-record T_L ──────────────────────────────────────────
        SE_fullTL     = its.standard_error_saunders(
            sigma_mo, fr['T_L_s'], T_record_mo)
        N_star_fullTL = its.effective_dof(len(x_clean), dt_s, fr['T_int_s'])

        # ── Monthly T_L with fallback ─────────────────────────────────────────
        nlags_mo  = min(int(MAX_LAG_DAYS * 24 / dt_hours), len(x_clean) // 2)
        rho_mo    = its.autocorrelation(x_clean, nlags=nlags_mo)
        zero_flag = False

        try:
            T_int_mo, k_star_mo = its.integral_timescale(rho_mo, dt=dt_s)
            # Guard: T_int > 80% of monthly record is unreliable
            if T_int_mo >= T_record_mo * 0.8:
                print(f"  {label}: T_int ({T_int_mo/86400:.1f}d) ≥ 80% "
                      f"of record — using full-record T_L")
                T_int_mo  = fr['T_int_s']
                k_star_mo = fr['k_star']
                zero_flag = True
        except ValueError:
            print(f"  {label}: ACF no zero crossing — using full-record T_L")
            T_int_mo  = fr['T_int_s']
            k_star_mo = fr['k_star']
            zero_flag = True

        T_L_mo         = its.one_sided_integral_timescale(T_int_mo)
        SE_monthlyTL   = its.standard_error_saunders(
            sigma_mo, T_L_mo, T_record_mo)
        N_star_monthly = its.effective_dof(len(x_clean), dt_s, T_int_mo)

        flag = " ⚠" if zero_flag else ""
        print(f"  {label}: T_L={T_L_mo/24:.1f}d  "
              f"N*_fullTL={N_star_fullTL:.0f}  SE_fullTL={SE_fullTL:.4f}  "
              f"N*_mo={N_star_monthly:.0f}  SE_mo={SE_monthlyTL:.4f}{flag}")

        records.append({
            'month':          label,
            'month_num':      month,
            'n':              len(x_clean),
            'sigma':          sigma_mo,
            'T_int_mo_h':     T_int_mo / 3600,
            'T_int_mo_d':     T_int_mo / 86400,
            'T_L_mo_h':       T_L_mo   / 3600,
            'T_L_mo_d':       T_L_mo   / 86400,
            'k_star_mo':      k_star_mo,
            'N_star_fullTL':  N_star_fullTL,
            'N_star_monthly': N_star_monthly,
            'SE_fullTL':      SE_fullTL,
            'SE_monthlyTL':   SE_monthlyTL,
            'rho_mo':         rho_mo,
            'dt_s':           dt_s,
            'fallback':       zero_flag,
        })

    monthly_results[name] = pd.DataFrame(records)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Summary data table
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*80}")
print("SUMMARY TABLE — Full record")
print(f"{'='*80}")
print(f"{'Mooring':<8} {'T_L (h)':>9} {'T_L (d)':>9} {'N*':>7} "
      f"{'sigma':>8} {'SE':>8}")
print(f"{'─'*54}")
for name, res in full_record_results.items():
    flag = " ⚠" if res['zero_cross_flag'] else ""
    print(f"{name:<8} {res['T_L_h']:>9.2f} {res['T_L_d']:>9.2f} "
          f"{res['N_star']:>7.1f} {res['sigma']:>8.4f} "
          f"{res['SE_full']:>8.4f}{flag}")

print(f"\n{'='*80}")
print("SUMMARY TABLE — Monthly SE comparison")
print(f"{'='*80}")

for name, df in monthly_results.items():
    fr = full_record_results[name]
    print(f"\n{name}  |  Full-record T_L = {fr['T_L_d']:.2f} d  "
          f"|  Full-record SE = {fr['SE_full']:.4f} cm/s")
    print(f"{'─'*72}")
    print(f"{'Month':<6} {'N':>6} {'sigma':>8} {'T_L_mo(d)':>11} "
          f"{'N*(fullTL)':>12} {'SE(fullTL)':>12} "
          f"{'N*(moTL)':>10} {'SE(moTL)':>10} {'Flag':>5}")
    print(f"{'─'*72}")
    for _, row in df.iterrows():
        flag = "⚠" if row['fallback'] else ""
        print(f"{row['month']:<6} {row['n']:>6} {row['sigma']:>8.4f} "
              f"{row['T_L_mo_d']:>11.2f} "
              f"{row['N_star_fullTL']:>12.1f} {row['SE_fullTL']:>12.4f} "
              f"{row['N_star_monthly']:>10.1f} {row['SE_monthlyTL']:>10.4f} "
              f"{flag:>5}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — ACF plots (full record + monthly overlaid)
# ═══════════════════════════════════════════════════════════════════════════════

for name, df in monthly_results.items():
    fr    = full_record_results[name]
    color = MOORINGS[name]['color']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Left: full record ACF ─────────────────────────────────────────────────
    ax      = axes[0]
    rho_f   = fr['rho']
    dt_h    = fr['dt_s'] / 3600
    lags_d  = np.arange(len(rho_f)) * dt_h / 24
    k_star  = fr['k_star']

    ax.plot(lags_d, rho_f, color=color, lw=2, label='Full record ACF')
    ax.fill_between(lags_d[:k_star+1], 0, rho_f[:k_star+1],
                    alpha=0.18, color=color,
                    label=f'T_L = {fr["T_L_d"]:.2f} d')
    ax.axvline(lags_d[min(k_star, len(lags_d)-1)],
               color=color, lw=1.5, ls='--', alpha=0.7,
               label=f'Zero crossing = {lags_d[min(k_star,len(lags_d)-1)]:.1f} d')
    ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
    ax.set_xlabel('Lag (days)', fontsize=11)
    ax.set_ylabel('Autocorrelation', fontsize=11)
    ax.set_ylim(-0.5, 1.05)
    ax.set_title(f'[A] Full record ACF — {name} u_prime',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── Right: monthly ACFs overlaid ──────────────────────────────────────────
    ax     = axes[1]
    cmap   = plt.cm.twilight
    n_mo   = len(df)
    colors = [cmap(i / max(n_mo-1, 1)) for i in range(n_mo)]

    for (_, row), c in zip(df.iterrows(), colors):
        rho_mo  = row['rho_mo']
        lags_mo = np.arange(len(rho_mo)) * (row['dt_s']/3600) / 24
        ax.plot(lags_mo, rho_mo, color=c, lw=1.5, alpha=0.85,
                label=f"{row['month']} (T_L={row['T_L_mo_d']:.1f}d)")

    ax.plot(lags_d, rho_f, color='k', lw=2.5, ls='--',
            label='Full record', zorder=5)
    ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
    ax.set_xlabel('Lag (days)', fontsize=11)
    ax.set_ylabel('Autocorrelation', fontsize=11)
    ax.set_ylim(-0.5, 1.05)
    ax.set_title(f'[B] Monthly ACFs — {name} u_prime',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right', ncol=2, framealpha=0.8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
#     plt.savefig(f'acf_{name}_u_prime.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — SE comparison plots
# ═══════════════════════════════════════════════════════════════════════════════

for name, df in monthly_results.items():
    fr    = full_record_results[name]
    color = MOORINGS[name]['color']
    mons  = df['month'].values
    x     = np.arange(len(mons))

    fig, axes = plt.subplots(3, 1, figsize=(10, 10),
                              gridspec_kw={'hspace': 0.4})

    # ── A: Monthly T_L vs full-record T_L ────────────────────────────────────
    ax = axes[0]
    ax.bar(x, df['T_L_mo_d'], color=color, alpha=0.7, label='Monthly T_L')
    ax.axhline(fr['T_L_d'], color='k', lw=2, ls='--',
               label=f'Full-record T_L = {fr["T_L_d"]:.2f} d')
    # Mark fallback months
    for i, (_, row) in enumerate(df.iterrows()):
        if row['fallback']:
            ax.text(i, df['T_L_mo_d'].max() * 0.95, '⚠',
                    ha='center', fontsize=10, color='gray')
    ax.set_xticks(x); ax.set_xticklabels(mons, fontsize=10)
    ax.set_ylabel('T_L (days)', fontsize=11)
    ax.set_title(f'[A] Monthly vs full-record T_L — {name} u_prime',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── B: N_eff comparison ───────────────────────────────────────────────────
    ax = axes[1]
    ax.bar(x - 0.2, df['N_star_fullTL'],  width=0.38,
           color='k', alpha=0.5, label='N* (full-record T_L)')
    ax.bar(x + 0.2, df['N_star_monthly'], width=0.38,
           color=color, alpha=0.7, label='N* (monthly T_L)')
    ax.set_xticks(x); ax.set_xticklabels(mons, fontsize=10)
    ax.set_ylabel('N* (effective DOF)', fontsize=11)
    ax.set_title(f'[B] N* comparison — {name} u_prime',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── C: SE comparison — key figure ────────────────────────────────────────
    ax = axes[2]
    ax.bar(x - 0.2, df['SE_fullTL'],    width=0.38,
           color='k', alpha=0.5,
           label='SE — full-record T_L')
    ax.bar(x + 0.2, df['SE_monthlyTL'], width=0.38,
           color=color, alpha=0.7,
           label='SE — monthly T_L')
    ax.axhline(fr['SE_full'], color='k', lw=2, ls=':',
               label=f'Full-record SE = {fr["SE_full"]:.4f} cm/s')
    ax.set_xticks(x); ax.set_xticklabels(mons, fontsize=10)
    ax.set_ylabel('Standard error (cm/s)', fontsize=11)
    ax.set_title(f'[C] SE comparison — {name} u_prime\n'
                 f'If bars similar → use full-record T_L for all months',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    fig.suptitle(f'Timescale and SE analysis — {name} u_prime',
                 fontsize=13, fontweight='bold')
    plt.savefig(f'timescale_SE_{name}_u_prime.png',
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Diagnostic: how variable are the monthly T_L estimates? ──────────────────
# High CV (coefficient of variation) = monthly T_L estimates are noisy
# = full-record T_L is more reliable

print("Stability of monthly T_L estimates:")
print(f"{'Mooring':<8} {'T_L_full (d)':>14} {'T_L_mo mean (d)':>17} "
      f"{'T_L_mo std (d)':>16} {'CV (%)':>8} {'Recommendation'}")
print('─' * 80)

for name, df in monthly_results.items():
    fr      = full_record_results[name]
    # Exclude fallback months from the monthly stats
    # (those just repeat the full-record value anyway)
    valid   = df[~df['fallback']]['T_L_mo_d']
    if len(valid) < 2:
        print(f"{name:<8}: insufficient non-fallback months")
        continue
    mo_mean = valid.mean()
    mo_std  = valid.std()
    cv      = 100 * mo_std / mo_mean

    if cv > 50:
        rec = "Use full-record T_L (monthly too noisy)"
    elif cv > 25:
        rec = "Either defensible — note in methods"
    else:
        rec = "Monthly T_L reasonable"

    print(f"{name:<8} {fr['T_L_d']:>14.2f} {mo_mean:>17.2f} "
          f"{mo_std:>16.2f} {cv:>8.1f}%  {rec}")

print()
print("CV > 50% → monthly T_L estimates vary too much month-to-month")
print("         → full-record T_L is more stable and defensible")
print("CV < 25% → monthly T_L is consistent → either approach justified")

In [ ]:
# ── Visual comparison: how much does the SE choice actually matter? ───────────
# If the two SE estimates are within ~30% of each other the choice
# makes little practical difference to your error bars

fig, axes = plt.subplots(1, len(monthly_results), figsize=(5*len(monthly_results), 5),
                          sharey=False)
if len(monthly_results) == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, monthly_results.items()):
    color = MOORINGS[name]['color']
    x     = np.arange(len(df))
    mons  = df['month'].values

    # Ratio of SE_fullTL / SE_monthlyTL
    # Values > 1 mean full-record is more conservative
    ratio = df['SE_fullTL'] / df['SE_monthlyTL']

    bars = ax.bar(x, ratio, color=color, alpha=0.7)
    ax.axhline(1.0, color='k', lw=1.5, ls='--', alpha=0.6,
               label='Equal SEs')
    ax.axhline(1.3, color='gray', lw=1, ls=':', alpha=0.6,
               label='30% more conservative')
    ax.axhline(2.0, color='gray', lw=1, ls=':', alpha=0.4,
               label='2× more conservative')

    # Annotate fallback months
    for i, (_, row) in enumerate(df.iterrows()):
        if row['fallback']:
            ax.text(i, ratio.iloc[i] + 0.02, '⚠',
                    ha='center', fontsize=9, color='gray')

    ax.set_xticks(x); ax.set_xticklabels(mons, fontsize=10)
    ax.set_ylabel('SE ratio (full-record TL / monthly TL)', fontsize=10)
    ax.set_title(f'{name}\nHow much more conservative is full-record T_L?',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('SE ratio — full-record T_L vs monthly T_L\n'
             'Ratio close to 1 → choice makes little difference',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('SE_ratio_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Diagnosing shit 

In [ ]:
# Quick isolation: which series is failing?
for name, ds in MOORING_DATASETS.items():
    x_raw = ds[VARNAME].values
    dt_seconds = float(ds['time'].diff('time').median().values / np.timedelta64(1, 's'))
    nlags_100d = int(100 * 86400 / dt_seconds)
    nlags_100d = min(nlags_100d, len(x_raw) - 1)

    rho_raw = its.autocorrelation(x_raw, nlags=nlags_100d)

    plt.figure(figsize=(6, 3))
    lags_days = np.arange(len(rho_raw)) * dt_seconds / 86400
    plt.plot(lags_days, rho_raw)
    plt.axhline(0, color='gray', linestyle='--', lw=0.7)
    plt.title(f'{name}: raw ACF out to 100 days')
    plt.xlabel('Lag [days]')
    plt.ylabel(r'$\rho$')
    plt.grid(alpha=0.3)
    plt.show()

    print(f"{name}: min rho over 100 days = {rho_raw.min():.4f}, "
          f"final rho at 100d = {rho_raw[-1]:.4f}")

In [ ]:
name = 'TNBD'  # or whichever mooring is failing
ds = MOORING_DATASETS[name]

dt_seconds = float(ds['time'].diff('time').median().values / np.timedelta64(1, 's'))
sampling_hours = dt_seconds / 3600.0

lp_ds, hp_ds = decompose_tidal_dataset(
    ds,
    var_names=[VARNAME],
    window_days=WINDOW_DAYS,
    sampling_hours=sampling_hours,
    window_type=WINDOW_TYPE,
)

x_lp = lp_ds[VARNAME].values
valid = np.isfinite(x_lp)
x_lp_clean = x_lp[valid]
t_lp_clean = ds['time'].values[valid]

# 1. Plot the actual low-passed time series (not the ACF) -- look for trend/drift
plt.figure(figsize=(10, 3))
plt.plot(t_lp_clean, x_lp_clean)
plt.axhline(0, color='gray', lw=0.7, linestyle='--')
plt.title(f'{name}: low-passed {VARNAME} time series')
plt.ylabel(VARNAME)
plt.grid(alpha=0.3)
plt.show()

# 2. ACF out to full record length (or as far as possible)
nlags_max = len(x_lp_clean) - 1
rho_lp = its.autocorrelation(x_lp_clean, nlags=nlags_max)

lags_days = np.arange(len(rho_lp)) * dt_seconds / 86400
plt.figure(figsize=(8, 3))
plt.plot(lags_days, rho_lp)
plt.axhline(0, color='gray', linestyle='--', lw=0.7)
plt.title(f'{name}: low-pass ACF, full record ({lags_days[-1]:.1f} days max)')
plt.xlabel('Lag [days]')
plt.ylabel(r'$\rho$')
plt.grid(alpha=0.3)
plt.show()

print(f"Record length: {lags_days[-1]:.1f} days")
print(f"min rho = {rho_lp.min():.4f}")
print(f"rho at 100 days = {rho_lp[np.argmin(np.abs(lags_days - 100))]:.4f}")

In [ ]:
TNBD_vel_bot

In [ ]:
import numpy as np

MOORING_DATASETS = {
    'TNBD': TNBD_vel_bot,
    'DITD': DITD_vel_top,
    'D':    D_vel_bot,
}

VARNAME = 'u'
WINDOW_DAYS = 3.0   # <-- set to whatever cutoff you're actually using
WINDOW_TYPE = 'cosine'

for name, ds in MOORING_DATASETS.items():

    dt_seconds = float(ds['time'].diff('time').median().values / np.timedelta64(1, 's'))

    # --- raw ---
    x_raw = ds[VARNAME].values
    rho_raw = its.autocorrelation(x_raw, nlags=None)
    T_int_raw, k_star_raw = its.integral_timescale(rho_raw, dt_seconds)

    # --- low-pass filtered ---
    lp_ds, hp_ds = decompose_tidal_dataset(
        ds,
        var_names=[VARNAME],
        window_days=WINDOW_DAYS,
        sampling_hours=dt_seconds / 3600.0,
        window_type=WINDOW_TYPE,
    )
    x_lp = lp_ds[VARNAME].values
    x_lp = x_lp[np.isfinite(x_lp)]  # drop filter edge NaNs

    rho_lp = its.autocorrelation(x_lp, nlags=None)
    T_int_lp, k_star_lp = its.integral_timescale(rho_lp, dt_seconds)

    print(f"{name}: T_int raw = {T_int_raw/86400:.2f} days, "
          f"T_int low-pass = {T_int_lp/86400:.2f} days "
          f"({'decreased' if T_int_lp < T_int_raw else 'increased'})")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MOORING_DATASETS = {
    'TNBD': TNBD_vel_bot,
    'DITD': DITD_vel_top,
    'D':    D_vel_bot,
}

VARNAME = 'u_prime'
WINDOW_DAYS = 3.0   # <-- set to whatever cutoff you're actually using
WINDOW_TYPE = 'cosine'

# Font sizes tuned for Overleaf (6.5in text width)
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7.5,
})

# ── COMPUTE AND STORE RESULTS ─────────────────────────────────────────────────
timescale_results = {}

for name, ds in MOORING_DATASETS.items():

    dt_seconds = float(ds['time'].diff('time').median().values / np.timedelta64(1, 's'))

    # --- raw ---
    x_raw = ds[VARNAME].values
    t_raw = ds['time'].values
    rho_raw = its.autocorrelation(x_raw, nlags=3000)
    T_int_raw, k_star_raw = its.integral_timescale(rho_raw, dt_seconds)
    T_L_raw = its.one_sided_integral_timescale(T_int_raw)

    # --- low-pass filtered ---
    lp_ds, hp_ds = decompose_tidal_dataset(
        ds,
        var_names=[VARNAME],
        window_days=WINDOW_DAYS,
        sampling_hours=dt_seconds / 3600.0,
        window_type=WINDOW_TYPE,
    )
    x_lp_full = lp_ds[VARNAME].values
    valid = np.isfinite(x_lp_full)
    x_lp = x_lp_full[valid]
    t_lp = t_raw[valid]

    rho_lp = its.autocorrelation(x_lp, nlags=3000)
    T_int_lp, k_star_lp = its.integral_timescale(rho_lp, dt_seconds)
    T_L_lp = its.one_sided_integral_timescale(T_int_lp)

    print(f"{name}: T_int raw = {T_L_raw/86400:.2f} days, "
          f"T_int low-pass = {T_L_lp/86400:.2f} days "
          f"({'decreased' if T_L_lp < T_L_raw else 'increased'})")

    timescale_results[name] = dict(
        dt_seconds=dt_seconds,
        x_raw=x_raw, t_raw=t_raw, rho_raw=rho_raw,
        T_int_raw=T_int_raw, k_star_raw=k_star_raw,
        x_lp=x_lp, t_lp=t_lp, rho_lp=rho_lp,
        T_int_lp=T_int_lp, k_star_lp=k_star_lp,
        T_L_raw=T_L_raw, T_L_lp=T_L_lp
    )

# # ── PANEL 1: ACF comparison (raw vs. low-pass), one panel per mooring ────────
# fig1, axes1 = plt.subplots(1, 3, figsize=(9, 3.2), sharey=True)

# for ax, (name, r) in zip(axes1, timescale_results.items()):
#     lags_days_raw = np.arange(len(r['rho_raw'])) * r['dt_seconds'] / 86400
#     lags_days_lp = np.arange(len(r['rho_lp'])) * r['dt_seconds'] / 86400

#     ax.plot(lags_days_raw, r['rho_raw'], color='k', lw=1, label='Raw')
#     ax.plot(lags_days_lp, r['rho_lp'], color='steelblue', lw=1, label='Low-pass')
#     ax.axhline(0, color='gray', lw=0.6, linestyle='--')
#     ax.set_title(name)
#     ax.set_xlabel('Lag [days]')
#     ax.grid(alpha=0.3)

#     T_raw_days = r['T_int_raw'] / 86400
#     T_lp_days = r['T_int_lp'] / 86400
#     ax.text(
#         0.97, 0.95,
#         f"$T_{{int}}$ raw = {T_raw_days:.2f} d\n"
#         f"$T_{{int}}$ LP = {T_lp_days:.2f} d",
#         transform=ax.transAxes,
#         ha='right', va='top',
#         fontsize=7,
#         bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='0.7')
#     )

# axes1[0].set_ylabel(r'$\rho$')
# axes1[0].legend(fontsize=7.5)

# # fig1.suptitle(r"Autocorrelation of $u'$: Raw vs. Low-pass", y=1.03)
# plt.tight_layout()
# plt.savefig('acf_comparison_3panel.png', dpi=300, bbox_inches='tight')
# plt.show()

# ── PANEL 2: u_prime time series comparison (raw vs. low-pass) ──────────────
fig2, axes2 = plt.subplots(3, 1, figsize=(6.5, 7), sharex=False)

for ax, (name, r) in zip(axes2, timescale_results.items()):
    ax.plot(r['t_raw'], r['x_raw'], color='0.7', lw=0.6, label='Raw', alpha=0.8)
    ax.plot(r['t_lp'], r['x_lp'], color='steelblue', lw=1.2, label='Low-pass')
    ax.axhline(0, color='gray', lw=0.6, linestyle='--')
    ax.set_title(name)
    ax.set_ylabel(r"u_prime [m s$^{-1}$]")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7.5, loc='upper right')

axes2[-1].set_xlabel('Time')

# fig2.suptitle(r"$u'$ Velocity: Raw vs. Low-pass", y=1.01)
plt.tight_layout()
plt.savefig('u_prime_comparison_3panel.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Font sizes tuned for Overleaf (6.5in text width)
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7.5,
})

# ── ACF comparison, stacked vertically (3 rows x 1 column) ──────────────────
fig1, axes1 = plt.subplots(3, 1, figsize=(6.5, 7), sharex=False)

for ax, (name, r) in zip(axes1, timescale_results.items()):
    lags_days_raw = np.arange(len(r['rho_raw'])) * r['dt_seconds'] / 86400
    lags_days_lp = np.arange(len(r['rho_lp'])) * r['dt_seconds'] / 86400

    ax.plot(lags_days_raw, r['rho_raw'], color='k', lw=1, label='Raw')
    ax.plot(lags_days_lp, r['rho_lp'], color='steelblue', lw=1, label='Low-pass')
    ax.axhline(0, color='gray', lw=0.6, linestyle='--')
    ax.set_title(name)
    ax.set_ylabel(r'ACF')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7.5)

    T_raw_days = r['T_L_raw'] / 86400
    T_lp_days = r['T_L_lp'] / 86400
    ax.text(
        0.97, 0.95,
        f"$T_{{int}}$ raw = {T_raw_days:.2f} d\n"
        f"$T_{{int}}$ LP = {T_lp_days:.2f} d",
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=7,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='0.7')
    )

axes1[-1].set_xlabel('Lag [days]')

# fig1.suptitle(r"Autocorrelation of $u'$: Raw vs. Low-pass", y=1.0)
plt.tight_layout()
plt.savefig('acf_comparison_3panel_u_prime.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
timescale_results.keys()

In [ ]:
timescale_results['TNBD'].keys()

In [ ]:
Tl_TNBD_u_prime = timescale_results['TNBD']['T_L_lp']
Tl_D_u_prime = timescale_results['D']['T_L_lp']
Tl_DITD_u_prime = timescale_results['DITD']['T_L_lp']

In [ ]:
Tl_TNBD_u_prime/86400, Tl_D_u_prime/86400, Tl_DITD_u_prime/86400

In [ ]:
Tl_TNBD_u/86400, Tl_D_u/86400, Tl_DITD_u/86400

In [ ]:
Tl_TNBD_v/86400, Tl_D_v/86400, Tl_DITD_v/86400

In [ ]:
TNBD_sigma_u = np.std(TNBD_vel_bot.u.values)
TNBD_sigma_v = np.std(TNBD_vel_bot.v.values)

D_sigma_u = np.std(D_vel_bot.u.values)
D_sigma_v = np.std(D_vel_bot.v.values)

DITD_sigma_u = np.std(DITD_vel_top.u.values)
DITD_sigma_v = np.std(DITD_vel_top.v.values)

In [ ]:
T_record = (len(TNBD_vel_bot.u) * 30) * 60

In [ ]:
T_record / 86400

In [ ]:
TNBD_u_SE = its.standard_error_saunders(TNBD_sigma_u, Tl_TNBD_u, T_record)
TNBD_v_SE = its.standard_error_saunders(TNBD_sigma_v, Tl_TNBD_v, T_record)

DITD_u_SE = its.standard_error_saunders(DITD_sigma_u, Tl_DITD_u, T_record)
DITD_v_SE = its.standard_error_saunders(DITD_sigma_v, Tl_DITD_v, T_record)

D_u_SE = its.standard_error_saunders(D_sigma_u, Tl_D_u, T_record)
D_v_SE = its.standard_error_saunders(D_sigma_v, Tl_D_v, T_record)

In [ ]:
TNBD_SE = np.sqrt(TNBD_u_SE**2 + TNBD_v_SE**2)
DITD_SE = np.sqrt(DITD_u_SE**2 + DITD_v_SE**2)
D_SE = np.sqrt(D_u_SE**2 + D_v_SE**2)


In [ ]:
TNBD_SE, DITD_SE, D_SE

In [ ]:
DITD

In [ ]:
its.standard_error_saunders?